# Claude Agent sample using Amazon Bedrock models
* **Host:** AgentCore Runtime
* **Instrumentation:** OpenInference
* **Observability:** Arize

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Claude Agents SDK
* Docker/Finch running
* Amazon CloudWatch Access
* Enable [transaction search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) on Amazon CloudWatch. 


In [ ]:
# !pip install --force-reinstall -U -r requirements-dev.txt --quiet
!pip install --force-reinstall -U -r requirements-arize.txt --quiet

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [ ]:
%%writefile claude_agent.py
"""Sample Claude Agent with Bedrock"""
import os
from claude_agent_sdk import (
    ClaudeAgentOptions,
    query,
    tool,
    create_sdk_mcp_server,
    ResultMessage,
)
from openinference.instrumentation.claude_agent_sdk import ClaudeAgentSDKInstrumentor
from opentelemetry import trace
# from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
# from opentelemetry.sdk import trace as trace_sdk
# from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from arize.otel import register
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Enable Bedrock mode
os.environ["CLAUDE_CODE_USE_BEDROCK"] = "1"
os.environ["AWS_REGION"] = "us-east-1"
os.environ["ANTHROPIC_MODEL"] = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Configure OpenInference instrumentation for Claude Agent SDK
if os.getenv("DISABLE_ADOT_OBSERVABILITY"):
    tracer_provider = register(
        # set_global_tracer_provider=False,
        log_to_console=True,
    )
    # tracer_provider = trace_sdk.TracerProvider()
    # tracer_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))
else:
    tracer_provider = trace.get_tracer_provider()
ClaudeAgentSDKInstrumentor().instrument(tracer_provider=tracer_provider)

# Init server
app = BedrockAgentCoreApp()


# Create a custom tool
@tool("add", "Add two numbers", {"a": float, "b": float})
async def add_numbers(args):
    """add two numbers"""
    result = args["a"] + args["b"]
    return {"content": [{"type": "text", "text": f"Result: {result}"}]}

# Create an SDK MCP server
server = create_sdk_mcp_server(
    name="my-tools",
    version="1.0.0",
    tools=[add_numbers]
)

# Use it with Claude
options = ClaudeAgentOptions(
    mcp_servers={"tools": server},
    allowed_tools=["mcp__tools__add"],
    output_format="json"
)


async def invoke_agent(user_query):
    """Invoke Claude Agent"""
    # async with ClaudeSDKClient(options=options) as client:
    #     await client.query(user_query)
    #     async for message in client.receive_response():
    #         if isinstance(message, ResultMessage):
    #             yield message.result
    async for message in query(
        prompt=user_query,
        options=options,
    ):
        if isinstance(message, ResultMessage):
            yield message.result

@app.entrypoint
async def claude_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_query = payload.get("prompt")
    print("User input:", user_query)
    
    response = ""
    async for result in invoke_agent(user_query):
        response += result
    return response

if __name__ == "__main__":
    app.run()

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

When configuring for containerized environment (such as docker) add the following command, an example is given below:

`CMD ["opentelemetry-instrument", "python", "claude_agent.py"]`


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "ac_runtime_claude_agent_sdk_obsy_demo"
response = agentcore_runtime.configure(
    entrypoint="claude_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    disable_otel=True
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
import base64
from dotenv import dotenv_values

config = dotenv_values(".env")

# Langfuse configuration
# otel_endpoint = config.get("LANGFUSE_OTEL_ENDPOINT", "https://us.cloud.langfuse.com/api/public/otel")
# langfuse_secret_key = config.get("LANGFUSE_SECRET_KEY", "")  # For production: key should be securely stored
# langfuse_public_key = config.get("LANGFUSE_PUBLIC_KEY", "")  # For production: key should be securely stored
# langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
# otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"

# Arize Phoenix configuration
arize_otel_endpoint = config.get("ARIZE_COLLECTOR_ENDPOINT", "https://app.arize.arize.com")
arize_api_key = config.get("ARIZE_API_KEY", "")  # For production key should be securely stored
arize_space_id = config.get("ARIZE_SPACE_ID", "")
arize_project_name = config.get("ARIZE_PROJECT_NAME", "")
otel_auth_header = f"Authorization=Bearer {arize_api_key}"
otel_resource= f"openinference.project.name={arize_project_name}"


launch_result = agentcore_runtime.launch(
    env_vars={
        "DISABLE_ADOT_OBSERVABILITY": "true",
        # "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use 3P OTEL endpoint
        # "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add 3P OTEL auth header
        "ARIZE_API_KEY": arize_api_key,  # Add Arize API Key
        "ARIZE_SPACE_ID": arize_space_id,  # Add Arize space id
        "ARIZE_PROJECT_NAME": arize_project_name,  # Add Arize project name
    }
)
launch_result

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
!agentcore status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
!agentcore invoke '{"prompt": "what is 400 + 200?"}'

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
!agentcore destroy --delete-ecr-repo --force --dry-run


# Congratulations!